# Stage E — Basin Discharge Modeling (Daily, Exogenous-only)

**Goal.** Train first-cut ML models to predict **daily basin discharge** from
**ERA5 basin-aggregated features** + **static basin attributes** (no discharge
history yet). Evaluate against strong baselines and pick a champion per basin.

---

## Inputs

- **Features (daily):**
  `data/modeling/features/era5_features_basin_daily.parquet`
- **Targets (daily discharge):**
  `data/basin_discharge/processed/station=<name>.parquet`
  + station↔basin mapping:  
  `data/modeling/targets/meta/station_to_basin_name.csv`
- (Derived in Stage D) **Joined training table (exogenous):**  
  `data/modeling/datasets/train_basin_daily_exogenous.parquet`

---

## Outputs (created in this notebook)

- **Metrics tables**
  - Baselines, Ridge, Tree: `data/modeling/reports/stageE_*_metrics.csv`
- **Predictions (optional)**
  - `data/modeling/predictions/exogenous/predictions_test.parquet`
- **Figures for slides**
  - `data/modeling/reports/figs/nse_by_basin_{val,test}.png`
  - `data/modeling/reports/figs/hydrograph_basin{B}_{YEAR}.png`
  - `data/modeling/reports/figs/residuals_by_month_basin{B}.png`

---

## Pipeline (what each step does)

1. **Load & sanity-check the training table**
   - Ensure expected columns: `['basin_id','date_local','discharge_cms', … features …]`
   - Report dtype mix, duplicate keys, per-basin date coverage, and % missing.

2. **Feature screening (numeric only)**
   - Remove **constant** columns.
   - Drop features with **>15% missing** (adjustable).
   - Keep the cleaned numeric feature list `FEAT_NUMERIC`.

3. **Time-ordered splits (per basin)**
   - For each basin: `train → val → test` (no overlap, no leakage).
   - Concatenate across basins to form pooled `TRAIN / VAL / TEST`.

4. **Baselines**
   - **Climatology** (train-mean).
   - **Persistence** (ŷt = yt-1).
   - Compute metrics on VAL/TEST: `MAE, RMSE, R², NSE, KGE`.

5. **Metric helpers**
   - Reusable functions to score arrays/DataFrames and to format tables.

6. **Pooled linear model (Ridge)**
   - Fit on **log1p(discharge)** with **StandardScaler**.
   - Small α grid; select **best by VAL NSE**.
   - Evaluate on VAL/TEST and store predictions.

7. **Pooled tree model**
   - Try LightGBM → XGBoost → **RandomForest** (fallback), all on **log1p(discharge)**.
   - Light tuning, select **best by VAL NSE**, evaluate VAL/TEST.

8. **Comparison & champion selection**
   - Merge all metrics; build **VAL/TEST NSE** pivots.
   - Compute **skill vs persistence** (NSE gain, RMSE reduction).
   - Pick **champion per basin** = highest **VAL NSE** (tie-break: lower RMSE).
   - Report **TEST** metrics for the champions.

9. **Presentation figures**
   - **Bars**: NSE by basin & model (VAL/TEST).
   - **Hydrographs**: Observed vs predicted (TEST) for each basin.
   - **Residual seasonality**: residuals by month (TEST).

---

## Key choices / assumptions

- Daily horizon; **exogenous-only** (no lagged discharge).
- Target transformed with **log1p**, predictions **expm1** then clipped ≥ 0.
- Splits are **chronological within basin**; metrics reported by basin and overall.

---

## Next upgrades (post-presentation)

- Add **ARX** (lagged discharge) and/or **per-basin models**.
- Try gradient boosting (LightGBM/XGBoost) with a slightly broader search.
- Simple **bias correction** on VAL (linear or quantile mapping).
- Event-oriented scores for flood relevance (hits/false alarms/CSI).


## Step 1: Load & sanity-check

In [17]:
# === Resolve Project Root ===
from pathlib import Path
import subprocess

def get_project_root(max_up=6):
    try:
        root = subprocess.check_output(["git","rev-parse","--show-toplevel"], text=True).strip()
        if root:
            return Path(root)
    except Exception:
        pass
    p = Path.cwd()
    for _ in range(max_up):
        if (p/"data").exists() and (p/"code").exists():
            return p
        if (p/".git").exists():
            return p
        p = p.parent
    return Path.cwd()

PROJECT_ROOT = get_project_root()
print("Project root:", PROJECT_ROOT)

EXOG_PATH = PROJECT_ROOT / "data/modeling/datasets/train_basin_daily_exogenous.parquet"
ARX_PATH  = PROJECT_ROOT / "data/modeling/datasets/train_basin_daily_arx.parquet"
print("Exogenous path:", EXOG_PATH)
print("ARX path:", ARX_PATH)

Project root: /Users/liuq13/bhutan_climate_modeling
Exogenous path: /Users/liuq13/bhutan_climate_modeling/data/modeling/datasets/train_basin_daily_exogenous.parquet
ARX path: /Users/liuq13/bhutan_climate_modeling/data/modeling/datasets/train_basin_daily_arx.parquet


## Small helpers for loading & summarizing

In [18]:
# --- Step 1.2: Helpers (load + summarize) ---
import pandas as pd
import numpy as np

def load_parquet_normalized(path: Path) -> pd.DataFrame:
    """Load parquet and normalize date_local to naive midnight; coerce basin_id to int where possible."""
    df = pd.read_parquet(path)
    if "date_local" in df.columns:
        df["date_local"] = pd.to_datetime(df["date_local"]).dt.tz_localize(None).dt.normalize()
    if "basin_id" in df.columns:
        # coerce safely; if there are NaN, leave them as Int64 (nullable)
        try:
            df["basin_id"] = pd.to_numeric(df["basin_id"], errors="coerce").astype("Int64")
        except Exception:
            pass
    return df

def summarize_dataset(name: str, df: pd.DataFrame, key=("basin_id","date_local")):
    print(f"\n=== [{name}] shape: {df.shape} ===")
    # dtypes overview
    print("\nDtype counts:")
    print(df.dtypes.value_counts())

    # columns by dtype
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    dt_cols  = [c for c in df.columns if pd.api.types.is_datetime64_any_dtype(df[c])]
    obj_cols = [c for c in df.columns if df[c].dtype == "object"]
    print(f"\nNumeric cols ({len(num_cols)}):", num_cols[:15], "..." if len(num_cols)>15 else "")
    print(f"Datetime cols ({len(dt_cols)}):", dt_cols)
    print(f"Object cols ({len(obj_cols)}):", obj_cols)

    # key missingness
    for k in ["discharge_cms","qc_any"]:
        if k in df.columns:
            frac = df[k].isna().mean()*100
            print(f"Missing {k}: {frac:.2f}%")

    # duplicate (basin_id, date_local)
    if all(k in df.columns for k in key):
        dup = df.duplicated(list(key)).sum()
        print(f"Duplicate ({key[0]},{key[1]}) rows:", dup)

    # per-basin summary
    if "basin_id" in df.columns and "date_local" in df.columns:
        g = (df.groupby("basin_id", dropna=False)
               .agg(first_day=("date_local","min"),
                    last_day =("date_local","max"),
                    n_days   =("date_local","nunique"),
                    n_rows   =("date_local","size"),
                    miss_y   =("discharge_cms", lambda s: int(s.isna().sum()) if "discharge_cms" in df.columns else np.nan))
               .reset_index()
               .sort_values("basin_id"))
        print("\nPer-basin date coverage:")
        display(g.head(20))
        print("Total basins:", g["basin_id"].nunique())

    # quick peek
    print("\nHead:")
    display(df.head(3))


## Execute Step 1 — load & inspect both files

In [20]:
# --- Step 1.3: Run sanity checks (no filtering yet) ---
exog_df = load_parquet_normalized(EXOG_PATH)  # raises if missing
summarize_dataset("EXOGENOUS", exog_df)


=== [EXOGENOUS] shape: (36924, 112) ===

Dtype counts:
float64           102
int64               4
object              2
Int64               1
datetime64[ns]      1
int32               1
float32             1
Name: count, dtype: int64

Numeric cols (109): ['basin_id', 'potential_evaporation_sum_1d', 'potential_evaporation_sum_3d', 'potential_evaporation_sum_7d', 'potential_evaporation_sum_14d', 'potential_evaporation_sum_30d', 'precipitation_sum_1d', 'precipitation_sum_3d', 'precipitation_sum_7d', 'precipitation_sum_14d', 'precipitation_sum_30d', 'runoff_sum_1d', 'runoff_sum_3d', 'runoff_sum_7d', 'runoff_sum_14d'] ...
Datetime cols (1): ['date_local']
Object cols (2): ['basin_name', 'qc_any']
Missing discharge_cms: 19.30%
Missing qc_any: 19.19%
Duplicate (basin_id,date_local) rows: 0

Per-basin date coverage:


,basin_id,first_day,last_day,n_days,n_rows,miss_y
0,3,1991-04-22,2024-12-31,12308,12308,3542
1,6,1991-04-22,2024-12-31,12308,12308,0
2,8,1991-04-22,2024-12-31,12308,12308,3584


Total basins: 3

Head:


,basin_id,date_local,potential_evaporation_sum_1d,potential_evaporation_sum_3d,potential_evaporation_sum_7d,potential_evaporation_sum_14d,potential_evaporation_sum_30d,precipitation_sum_1d,precipitation_sum_3d,precipitation_sum_7d,...,log_acc_mean,log_acc_max,pct_slope_gt_a,pct_slope_gt_b,acc_mean,acc_max,feature_na_count,feature_na_frac,discharge_cms,qc_any
0,3,1991-04-22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,6.976817,13.794746,77.342031,25.226937,1070.502172,979449.0,77,0.726415,NaN,None
1,3,1991-04-23,-0.000256,NaN,NaN,NaN,NaN,0.001312,NaN,NaN,...,6.976817,13.794746,77.342031,25.226937,1070.502172,979449.0,58,0.547170,NaN,None
2,3,1991-04-24,-0.000298,NaN,NaN,NaN,NaN,0.002204,NaN,NaN,...,6.976817,13.794746,77.342031,25.226937,1070.502172,979449.0,58,0.547170,NaN,None


In [22]:
# ARX may be absent; handle gracefully
try:
    arx_df = load_parquet_normalized(ARX_PATH)
    summarize_dataset("ARX", arx_df)
except FileNotFoundError:
    arx_df = None
    print("\n[WARN] ARX parquet not found; we can proceed exogenous-only and add ARX later.")


=== [ARX] shape: (29666, 120) ===

Dtype counts:
float64           104
float32             7
int64               4
Int64               1
datetime64[ns]      1
int32               1
object              1
bool                1
Name: count, dtype: int64

Numeric cols (118): ['basin_id', 'potential_evaporation_sum_1d', 'potential_evaporation_sum_3d', 'potential_evaporation_sum_7d', 'potential_evaporation_sum_14d', 'potential_evaporation_sum_30d', 'precipitation_sum_1d', 'precipitation_sum_3d', 'precipitation_sum_7d', 'precipitation_sum_14d', 'precipitation_sum_30d', 'runoff_sum_1d', 'runoff_sum_3d', 'runoff_sum_7d', 'runoff_sum_14d'] ...
Datetime cols (1): ['date_local']
Object cols (1): ['basin_name']
Missing discharge_cms: 0.00%
Missing qc_any: 0.00%
Duplicate (basin_id,date_local) rows: 0

Per-basin date coverage:


,basin_id,first_day,last_day,n_days,n_rows,miss_y
0,3,2000-01-31,2023-12-31,8736,8736,0
1,6,1991-05-22,2024-12-31,12278,12278,0
2,8,2000-01-31,2023-12-31,8652,8652,0


Total basins: 3

Head:


,basin_id,date_local,potential_evaporation_sum_1d,potential_evaporation_sum_3d,potential_evaporation_sum_7d,potential_evaporation_sum_14d,potential_evaporation_sum_30d,precipitation_sum_1d,precipitation_sum_3d,precipitation_sum_7d,...,discharge_cms,qc_any,q_lag_1d,q_lag_2d,q_lag_3d,q_lag_7d,q_lag_14d,q_lag_30d,q_roll7_mean,q_roll14_std
0,3,2000-01-31,-0.000218,-0.000624,-0.001396,-0.002778,-0.006060,0.000112,0.001292,0.002773,...,11.928,False,11.590,12.506,13.302,13.099,13.715,14.783,12.885714,0.568484
1,3,2000-02-01,-0.000181,-0.000598,-0.001385,-0.002764,-0.006019,0.000428,0.001114,0.002549,...,12.550,False,11.928,11.590,12.506,13.302,13.715,14.620,12.718429,0.627722
2,3,2000-02-02,-0.000175,-0.000574,-0.001364,-0.002677,-0.005945,0.000430,0.000970,0.002714,...,12.160,False,12.550,11.928,11.590,13.099,13.872,14.565,12.611000,0.604644


## Apply data policy & define features

In [23]:
# --- Step 2.1: Coerce qc_any -> boolean and filter rows ---

import numpy as np
import pandas as pd

def to_bool_series(s: pd.Series) -> pd.Series:
    """Robust bool coercion (returns pandas Boolean dtype)."""
    if s.dtype == bool:
        return s.astype("boolean")
    truthy = {"true","t","yes","y","1",1,True}
    falsy  = {"false","f","no","n","0",0,False}
    def _cast(v):
        if pd.isna(v):
            return pd.NA
        v2 = str(v).strip().lower()
        if v2 in truthy: return True
        if v2 in falsy:  return False
        return pd.NA
    return s.map(_cast).astype("boolean")

# Work on a copy of the exogenous dataset first
work = exog_df.copy()

# Coerce qc_any -> boolean (create if absent, default False = keep)
if "qc_any" in work.columns:
    work["qc_any"] = to_bool_series(work["qc_any"])
else:
    work["qc_any"] = pd.Series(False, index=work.index, dtype="boolean")

# Ensure basin_id is integer-like
work["basin_id"] = pd.to_numeric(work["basin_id"], errors="coerce").astype("Int64")

# Policy: keep rows with qc_any == False AND non-null discharge
mask_keep = (work["qc_any"] == False) & (work["discharge_cms"].notna())
kept_rows = mask_keep.sum()
print(f"Rows kept after policy: {kept_rows} / {len(work)} ({kept_rows/len(work):.1%})")
exog_clean = work.loc[mask_keep].copy()

# Sanity peek
exog_clean[["basin_id","date_local","discharge_cms","qc_any"]].head()


Rows kept after policy: 29143 / 36924 (78.9%)


,basin_id,date_local,discharge_cms,qc_any
3176,3,2000-01-01,14.783,False
3177,3,2000-01-02,14.620,False
3178,3,2000-01-03,14.565,False
3179,3,2000-01-04,14.031,False
3180,3,2000-01-05,14.032,False


In [24]:
# --- Step 2.2: Build numeric-only feature list and report missingness/constancy ---

RESERVED = {"basin_id","date_local","discharge_cms","qc_any","basin_name"}  # add any other non-features here

# numeric candidate features (exclude reserved)
num_candidates = [c for c in exog_clean.columns
                  if c not in RESERVED and pd.api.types.is_numeric_dtype(exog_clean[c])]

print(f"Numeric candidate features: {len(num_candidates)}")

# Report missing fraction & constant flags (on the filtered data)
miss_frac = exog_clean[num_candidates].isna().mean()
is_const  = exog_clean[num_candidates].nunique(dropna=True) <= 1

feature_report = (pd.DataFrame({
    "missing_frac": miss_frac,
    "is_constant":  is_const
}).sort_values(["is_constant","missing_frac"], ascending=[False, False]))

display(feature_report.head(20))
print("Any all-NaN features:", int((miss_frac==1.0).sum()))
print("Any constant features:", int(is_const.sum()))


Numeric candidate features: 107


,missing_frac,is_constant
low_coverage_mean_14d,0.000480,True
low_coverage_mean_7d,0.000240,True
low_coverage_mean_3d,0.000103,True
low_coverage_mean_1d,0.000034,True
potential_evaporation_sum_30d,0.001029,False
precipitation_sum_30d,0.001029,False
runoff_sum_30d,0.001029,False
snowmelt_sum_30d,0.001029,False
solar_radiation_sum_30d,0.001029,False
sub_surface_runoff_sum_30d,0.001029,False


Any all-NaN features: 0
Any constant features: 4


In [25]:
# --- Step 2.3: Drop by simple rules (adjustable) ---

NA_THRESH = 0.15  # 15% missing allowed; drop if > NA_THRESH

FEAT_NUMERIC = [c for c in num_candidates
                if (miss_frac[c] <= NA_THRESH) and (not is_const[c])]

dropped = sorted(set(num_candidates) - set(FEAT_NUMERIC))
print(f"Kept features: {len(FEAT_NUMERIC)}  |  Dropped: {len(dropped)}")

# Small summary to confirm shapes for next step
print("exog_clean shape:", exog_clean.shape)
print("First 5 kept features:", FEAT_NUMERIC[:5])


Kept features: 103  |  Dropped: 4
exog_clean shape: (29143, 112)
First 5 kept features: ['potential_evaporation_sum_1d', 'potential_evaporation_sum_3d', 'potential_evaporation_sum_7d', 'potential_evaporation_sum_14d', 'potential_evaporation_sum_30d']


## Step 3: per-basin time splits (70/15/15)

In [26]:
# --- Step 3.1: Add chronological train/val/test splits per basin ---

import numpy as np
import pandas as pd

def add_time_splits(
    df: pd.DataFrame,
    id_col: str = "basin_id",
    date_col: str = "date_local",
    rule=(0.70, 0.15, 0.15),
) -> pd.DataFrame:
    """
    Label each row with 'train'/'val'/'test' per basin using chronological splits.
    Splits are based on the sorted UNIQUE dates within each basin.
    """
    out = df.copy()
    out["split"] = pd.NA

    for b, g in out.sort_values([id_col, date_col]).groupby(id_col, sort=False):
        dates = np.array(sorted(g[date_col].dropna().unique()))
        n = len(dates)
        if n < 10:
            out.loc[g.index, "split"] = "train"
            continue

        # compute cut sizes (ensure at least 1 test day)
        n_train = int(np.floor(rule[0] * n))
        n_val   = int(np.floor(rule[1] * n))
        if n_train < 1: n_train = 1
        if n_val   < 1: n_val = 1
        if n_train + n_val >= n:  # leave at least 1 for test
            n_val = max(1, n - n_train - 1)
        n_test  = n - n_train - n_val
        if n_test < 1:
            n_test = 1
            if n_val > 1:
                n_val -= 1

        cut_train = dates[n_train - 1]
        cut_val   = dates[n_train + n_val - 1]

        idx = g.index
        d   = out.loc[idx, date_col]
        out.loc[idx, "split"] = np.where(
            d <= cut_train, "train",
            np.where(d <= cut_val, "val", "test")
        )

    return out

# Apply to cleaned exogenous table from Step 2
exog_split = add_time_splits(exog_clean, id_col="basin_id", date_col="date_local", rule=(0.70,0.15,0.15))
print("Null splits:", int(exog_split["split"].isna().sum()))


Null splits: 0


In [27]:
# --- Step 3.2: Summaries for splits ---

# counts (rows) by basin × split
count_tbl = (exog_split
             .groupby(["basin_id","split"], dropna=False)
             .size()
             .unstack("split", fill_value=0)
             .reset_index()
             .sort_values("basin_id"))
display(count_tbl)

# date ranges by basin × split
span_tbl = (exog_split
            .groupby(["basin_id","split"])
            .agg(first=("date_local","min"), last=("date_local","max"))
            .unstack("split"))
# tidy column names
span_tbl.columns = [f"{sp}_{which}" for which, sp in span_tbl.columns]
span_tbl = span_tbl.reset_index().sort_values("basin_id")
display(span_tbl)


split,basin_id,test,train,val
0,3,1289,6008,1287
1,6,1796,8374,1794
2,8,1290,6016,1289


,basin_id,test_first,train_first,val_first,test_last,train_last,val_last
0,3,2020-05-30,2000-01-01,2016-10-22,2023-12-31,2016-10-21,2020-05-29
1,6,2019-12-01,1991-04-22,2014-11-16,2024-12-31,2014-11-15,2019-11-30
2,8,2020-04-29,2000-01-01,2016-09-20,2023-12-31,2016-09-19,2020-04-28


## Step 4: build NaN-free design matrices (median fill + stable one-hot)

In [28]:
# --- Step 4.1: helpers to one-hot basin_id and to build matrices ---

import numpy as np
import pandas as pd

def onehot_from_categories(s: pd.Series, categories: list, prefix="basin_id") -> pd.DataFrame:
    """
    Stable one-hot using TRAIN categories.
    Any category not in `categories` is ignored; missing columns are filled with zeros.
    """
    s = pd.to_numeric(s, errors="coerce").astype("Int64")
    cols = [f"{prefix}_{c}" for c in categories]
    oh = pd.DataFrame(0, index=s.index, columns=cols, dtype=np.float32)
    mask = s.notna()
    if mask.any():
        dmy = pd.get_dummies(s[mask].astype("int64"))
        dmy.columns = [f"{prefix}_{c}" for c in dmy.columns]
        oh.loc[mask, dmy.columns] = dmy.values
    return oh

def build_design_matrices(df: pd.DataFrame,
                          feature_cols: list,
                          id_col="basin_id",
                          y_col="discharge_cms",
                          date_col="date_local"):
    """
    Returns:
      Xtr, ytr, Xva, yva, Xte, yte, feature_names (in order)
    """
    # Split views
    tr = df.loc[df["split"]=="train"].sort_values([id_col, date_col]).copy()
    va = df.loc[df["split"]=="val"  ].sort_values([id_col, date_col]).copy()
    te = df.loc[df["split"]=="test" ].sort_values([id_col, date_col]).copy()

    # Medians fit on TRAIN only
    med = tr[feature_cols].median()

    # Numeric block with median fill
    Xtr_num = tr[feature_cols].fillna(med)
    Xva_num = va[feature_cols].fillna(med)
    Xte_num = te[feature_cols].fillna(med)

    # Stable one-hot for basin_id using TRAIN categories
    cats = sorted(pd.to_numeric(tr[id_col], errors="coerce").dropna().astype("int64").unique().tolist())
    O_tr = onehot_from_categories(tr[id_col], cats, prefix=id_col)
    O_va = onehot_from_categories(va[id_col], cats, prefix=id_col)
    O_te = onehot_from_categories(te[id_col], cats, prefix=id_col)

    # Concatenate numeric + one-hot; align columns to TRAIN layout
    Xtr_df = pd.concat([Xtr_num.reset_index(drop=True), O_tr.reset_index(drop=True)], axis=1)
    Xva_df = pd.concat([Xva_num.reset_index(drop=True), O_va.reset_index(drop=True)], axis=1).reindex(columns=Xtr_df.columns, fill_value=0)
    Xte_df = pd.concat([Xte_num.reset_index(drop=True), O_te.reset_index(drop=True)], axis=1).reindex(columns=Xtr_df.columns, fill_value=0)

    # Final arrays and feature names
    featnames = Xtr_df.columns.tolist()
    Xtr = Xtr_df.to_numpy(dtype=float)
    Xva = Xva_df.to_numpy(dtype=float)
    Xte = Xte_df.to_numpy(dtype=float)
    ytr = tr[y_col].to_numpy()
    yva = va[y_col].to_numpy()
    yte = te[y_col].to_numpy()

    # Safety checks
    for name, X in [("train", Xtr), ("val", Xva), ("test", Xte)]:
        if np.isnan(X).any():
            raise ValueError(f"NaNs remain in {name} features.")
    assert len(ytr)==len(Xtr) and len(yva)==len(Xva) and len(yte)==len(Xte), "X/y length mismatch"

    # Small summary
    print(f"Numeric features kept: {len(feature_cols)} | Basin one-hot: {len(cats)} | Total features: {len(featnames)}")
    print(f"Shapes -> Xtr:{Xtr.shape}, Xva:{Xva.shape}, Xte:{Xte.shape}")

    return Xtr, ytr, Xva, yva, Xte, yte, featnames


In [29]:
# --- Step 4.2: build matrices for the EXOGENOUS track ---

Xtr_exog, ytr_exog, Xva_exog, yva_exog, Xte_exog, yte_exog, FEATNAMES_EXOG = build_design_matrices(
    exog_split,
    FEAT_NUMERIC,
    id_col="basin_id",
    y_col="discharge_cms",
    date_col="date_local",
)

# Extra verification
print("Any NaN in y? ->",
      np.isnan(ytr_exog).any() or np.isnan(yva_exog).any() or np.isnan(yte_exog).any())


Numeric features kept: 103 | Basin one-hot: 3 | Total features: 106
Shapes -> Xtr:(20398, 106), Xva:(4370, 106), Xte:(4375, 106)
Any NaN in y? -> False


/var/folders/br/nr4k1vxj1_j7jxk17x7xr8n9g2q0k8/T/ipykernel_84568/1775629320.py:18: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ True  True  True ... False False False]' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  oh.loc[mask, dmy.columns] = dmy.values
/var/folders/br/nr4k1vxj1_j7jxk17x7xr8n9g2q0k8/T/ipykernel_84568/1775629320.py:18: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[False False False ... False False False]' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  oh.loc[mask, dmy.columns] = dmy.values
/var/folders/br/nr4k1vxj1_j7jxk17x7xr8n9g2q0k8/T/ipykernel_84568/1775629320.py:18: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[False False False

## Step 5: baselines (climatology & persistence)

In [31]:
# --- Step 5.1: Metrics helpers (ignore NaN pairs) ---

import numpy as np
import pandas as pd

def _valid_pairs(y, yhat):
    y = np.asarray(y, dtype=float)
    yhat = np.asarray(yhat, dtype=float)
    m = ~(np.isnan(y) | np.isnan(yhat))
    return y[m], yhat[m]

def mae(y, yhat):
    y, yhat = _valid_pairs(y, yhat)
    return np.mean(np.abs(y - yhat))

def rmse(y, yhat):
    y, yhat = _valid_pairs(y, yhat)
    return float(np.sqrt(np.mean((y - yhat) ** 2)))

def r2(y, yhat):
    y, yhat = _valid_pairs(y, yhat)
    if y.size == 0:
        return np.nan
    ss_res = np.sum((y - yhat) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    return 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

def nse(y, yhat):
    # Nash–Sutcliffe Efficiency
    return r2(y, yhat)

def kge(y, yhat):
    # Kling–Gupta Efficiency (2009)
    y, yhat = _valid_pairs(y, yhat)
    if y.size == 0:
        return np.nan
    r = np.corrcoef(y, yhat)[0, 1] if np.std(y) > 0 and np.std(yhat) > 0 else np.nan
    alpha = np.std(yhat) / np.std(y) if np.std(y) > 0 else np.nan
    beta = np.mean(yhat) / np.mean(y) if np.mean(y) != 0 else np.nan
    if any(map(lambda v: np.isnan(v), [r, alpha, beta])):
        return np.nan
    return 1 - np.sqrt((r - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2)

def metric_row(y, yhat, extra: dict):
    return dict(
        **extra,
        n=len(_valid_pairs(y, yhat)[0]),
        mae=mae(y, yhat),
        rmse=rmse(y, yhat),
        r2=r2(y, yhat),
        nse=nse(y, yhat),
        kge=kge(y, yhat),
    )


### 5.2 Baselines (fit on TRAIN only; evaluate on VAL/TEST)

In [33]:
# --- Step 5.2: Baseline predictors ---

def fit_climatology(train_df: pd.DataFrame) -> dict:
    """
    Per-basin day-of-year median discharge.
    Fallbacks:
      - basin-wide median (train)
      - global median (train)
    Returns a dict with three Series/DataFrames to use in prediction.
    """
    tr = train_df.copy()
    tr["doy"] = tr["date_local"].dt.dayofyear.clip(upper=365)  # map 366 -> 365
    # per-basin-day median
    med_basin_doy = (tr.groupby(["basin_id", "doy"])["discharge_cms"]
                       .median().rename("q_med"))
    # per-basin overall median
    med_basin = tr.groupby("basin_id")["discharge_cms"].median().rename("q_med_basin")
    # global median
    med_global = float(tr["discharge_cms"].median())
    return {"by_basin_doy": med_basin_doy, "by_basin": med_basin, "global": med_global}

def predict_climatology(df_part: pd.DataFrame, model: dict) -> np.ndarray:
    part = df_part.copy()
    part["doy"] = part["date_local"].dt.dayofyear.clip(upper=365)

    # join basin-doy median
    key = ["basin_id", "doy"]
    pred = part.merge(model["by_basin_doy"].reset_index(), on=key, how="left")["q_med"]

    # fallback to basin median
    miss = pred.isna()
    if miss.any():
        pred.loc[miss] = part.loc[miss].merge(
            model["by_basin"].reset_index(), on="basin_id", how="left"
        )["q_med_basin"].values

    # fallback to global median
    pred.fillna(model["global"], inplace=True)
    return pred.to_numpy(dtype=float)

def predict_persistence(df_full: pd.DataFrame, df_part_idx: pd.Index) -> np.ndarray:
    """
    Persistence: y_hat(t) = y(t-1), computed within each basin on the full split table
    so the first VAL day can use the last TRAIN day (realistic at inference).
    """
    srt = df_full.sort_values(["basin_id", "date_local"]).copy()
    srt["y_pers"] = srt.groupby("basin_id")["discharge_cms"].shift(1)
    # pull back to the requested rows (VAL/TEST)
    return srt.loc[df_part_idx, "y_pers"].to_numpy(dtype=float)


## 5.3 Evaluate baselines on VAL & TEST (per-basin + overall)

In [36]:
# --- Step 5.3 (patched): Evaluate baselines per-basin + overall ---

def metrics_for_split(df_part: pd.DataFrame, yhat: np.ndarray,
                      track_name: str, baseline_name: str, split_label: str):
    """
    Build metric rows for each basin and an overall row, using df_part masks
    (avoids grouping mistakes on the Series).
    """
    rows = []
    assert len(df_part) == len(yhat), "Prediction length mismatch."

    # Per-basin rows
    for b in sorted(pd.to_numeric(df_part["basin_id"], errors="coerce").dropna().astype(int).unique()):
        msk = (df_part["basin_id"] == b)
        y_true_b = df_part.loc[msk, "discharge_cms"].values
        y_hat_b  = yhat[msk]
        rows.append(
            metric_row(
                y_true_b, y_hat_b,
                dict(track=track_name, model="baseline", baseline=baseline_name,
                     split=split_label, basin_id=int(b))
            )
        )

    # Overall row
    rows.append(
        metric_row(
            df_part["discharge_cms"].values, yhat,
            dict(track=track_name, model="baseline", baseline=baseline_name,
                 split=split_label, basin_id="ALL")
        )
    )
    return rows

def eval_baselines(df_split: pd.DataFrame, track_name="exogenous"):
    rows = []

    # Partitions
    train = df_split[df_split["split"] == "train"].copy()
    val   = df_split[df_split["split"] == "val"].copy()
    test  = df_split[df_split["split"] == "test"].copy()

    # Fit climatology on TRAIN
    clim = fit_climatology(train)

    # Climatology predictions
    yhat_val_clim  = predict_climatology(val,  clim)
    yhat_test_clim = predict_climatology(test, clim)

    # Persistence predictions (use full DF to allow train->val boundary lag)
    yhat_val_pers  = predict_persistence(df_split, val.index)
    yhat_test_pers = predict_persistence(df_split, test.index)

    # Collect metric rows
    rows += metrics_for_split(val,  yhat_val_clim,  track_name, "climatology", "val")
    rows += metrics_for_split(test, yhat_test_clim, track_name, "climatology", "test")
    rows += metrics_for_split(val,  yhat_val_pers,  track_name, "persistence", "val")
    rows += metrics_for_split(test, yhat_test_pers, track_name, "persistence", "test")

    metrics_df = pd.DataFrame(rows)[["track","model","baseline","split","basin_id","n","mae","rmse","r2","nse","kge"]]

    # Also return predictions if you want to plot later
    preds = {
        "val":  pd.DataFrame({"basin_id": val["basin_id"].values,
                              "date_local": val["date_local"].values,
                              "y_true": val["discharge_cms"].values,
                              "yhat_clim": yhat_val_clim,
                              "yhat_pers": yhat_val_pers}),
        "test": pd.DataFrame({"basin_id": test["basin_id"].values,
                              "date_local": test["date_local"].values,
                              "y_true": test["discharge_cms"].values,
                              "yhat_clim": yhat_test_clim,
                              "yhat_pers": yhat_test_pers}),
    }
    return metrics_df, preds

# Re-run baselines with the patched evaluator
baseline_metrics_exog, baseline_preds_exog = eval_baselines(exog_split, track_name="exogenous")
display(baseline_metrics_exog.sort_values(["split","basin_id","baseline"]))


,track,model,baseline,split,basin_id,n,mae,rmse,r2,nse,kge
4,exogenous,baseline,climatology,test,3,1289,10.031909,17.070635,0.850746,0.850746,0.916641
12,exogenous,baseline,persistence,test,3,1289,5.380677,10.823919,0.939994,0.939994,0.969980
5,exogenous,baseline,climatology,test,6,1796,5.112180,9.156939,0.769334,0.769334,0.850849
13,exogenous,baseline,persistence,test,6,1796,1.974102,4.673783,0.939907,0.939907,0.969952
6,exogenous,baseline,climatology,test,8,1290,70.045003,124.066135,0.805534,0.805534,0.838996
14,exogenous,baseline,persistence,test,8,1290,42.069159,82.063095,0.914919,0.914919,0.957451
7,exogenous,baseline,climatology,test,ALL,4375,25.707579,68.255688,0.889863,0.889863,0.881166
15,exogenous,baseline,persistence,test,ALL,4375,14.800091,45.046164,0.952030,0.952030,0.976013
0,exogenous,baseline,climatology,val,3,1287,8.509710,16.825163,0.866820,0.866820,0.859353
8,exogenous,baseline,persistence,val,3,1287,4.617256,10.148646,0.951545,0.951545,0.975773


## Step 6: pooled Ridge model on log1p(discharge)

### 6.1 Train pooled Ridge (log1p target) and choose alpha on VAL

In [37]:
# --- Step 6.1: Ridge (pooled) with log1p target; select alpha on VAL ---

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

def train_ridge_log1p(Xtr, ytr, Xva, yva, alphas=(0.1, 1, 3, 10, 30, 100, 300, 1000)):
    # scale features (fit on train only)
    scaler = StandardScaler(with_mean=True, with_std=True)
    Xtr_s = scaler.fit_transform(Xtr)
    Xva_s = scaler.transform(Xva)

    # transform target
    ytr_log = np.log1p(np.clip(ytr, 0, None))
    yva_true = yva.astype(float)

    scores = []
    best = None
    for a in alphas:
        mdl = Ridge(alpha=float(a), random_state=0)
        mdl.fit(Xtr_s, ytr_log)
        # predict & invert
        yva_hat = np.expm1(mdl.predict(Xva_s))
        yva_hat = np.clip(yva_hat, 0, None)
        # NSE (same as r2 here)
        score = nse(yva_true, yva_hat)
        scores.append((a, score))
        if (best is None) or (score > best[1]):
            best = (a, score, mdl)

    # Return best model + scaler + validation scores
    return {
        "alpha": best[0],
        "val_nse": best[1],
        "model": best[2],
        "scaler": scaler,
        "scores": pd.DataFrame(scores, columns=["alpha","val_nse"]).sort_values("alpha")
    }

ridge_search = train_ridge_log1p(Xtr_exog, ytr_exog, Xva_exog, yva_exog)
ridge_search["scores"]


,alpha,val_nse
0,0.1,0.932189
1,1.0,0.932144
2,3.0,0.932163
3,10.0,0.932200
4,30.0,0.931899
5,100.0,0.930073
6,300.0,0.925982
7,1000.0,0.918915


### 6.2 Evaluate on VAL & TEST (per-basin + overall) and return predictions

In [39]:
# --- Step 6.2: Evaluate pooled Ridge on VAL & TEST ---

# Recreate the sorted VAL/TEST slices so index alignment matches Step 4
VAL_DF  = exog_split.loc[exog_split["split"]=="val" ].sort_values(["basin_id","date_local"]).copy()
TEST_DF = exog_split.loc[exog_split["split"]=="test"].sort_values(["basin_id","date_local"]).copy()

def predict_with_ridge(ridge_pack, X):
    Xs = ridge_pack["scaler"].transform(X)
    yhat = np.expm1(ridge_pack["model"].predict(Xs))
    return np.clip(yhat, 0, None)

yhat_val_ridge  = predict_with_ridge(ridge_search, Xva_exog)
yhat_test_ridge = predict_with_ridge(ridge_search, Xte_exog)

def metrics_from_arrays(df_part, y_true, y_hat, track_name, model_name, split_label):
    rows = []
    # per-basin
    for b in sorted(pd.to_numeric(df_part["basin_id"], errors="coerce").dropna().astype(int).unique()):
        m = (df_part["basin_id"] == b)
        rows.append(
            metric_row(
                y_true[m], y_hat[m],
                dict(track=track_name, model=model_name, baseline="",
                     split=split_label, basin_id=int(b))
            )
        )
    # overall
    rows.append(
        metric_row(
            y_true, y_hat,
            dict(track=track_name, model=model_name, baseline="",
                 split=split_label, basin_id="ALL")
        )
    )
    return pd.DataFrame(rows)

ridge_metrics_val  = metrics_from_arrays(VAL_DF,  yva_exog,  yhat_val_ridge,
                                         "exogenous", "ridge_pooled_log1p", "val")
ridge_metrics_test = metrics_from_arrays(TEST_DF, yte_exog, yhat_test_ridge,
                                         "exogenous", "ridge_pooled_log1p", "test")
ridge_metrics = pd.concat([ridge_metrics_val, ridge_metrics_test], ignore_index=True)

print(f"Chosen alpha: {ridge_search['alpha']}  |  VAL NSE: {ridge_search['val_nse']:.3f}")
display(ridge_metrics.sort_values(["split","basin_id"]))


Chosen alpha: 10  |  VAL NSE: 0.932


,track,model,baseline,split,basin_id,n,mae,rmse,r2,nse,kge
4,exogenous,ridge_pooled_log1p,,test,3,1289,9.765049,16.163297,0.866191,0.866191,0.797522
5,exogenous,ridge_pooled_log1p,,test,6,1796,4.948941,8.430321,0.804489,0.804489,0.775466
6,exogenous,ridge_pooled_log1p,,test,8,1290,52.420569,91.185828,0.894951,0.894951,0.935221
7,exogenous,ridge_pooled_log1p,,test,ALL,4375,20.365253,50.575115,0.939531,0.939531,0.958443
0,exogenous,ridge_pooled_log1p,,val,3,1287,7.522768,13.871671,0.909473,0.909473,0.941212
1,exogenous,ridge_pooled_log1p,,val,6,1794,5.179499,8.701634,0.786226,0.786226,0.697025
2,exogenous,ridge_pooled_log1p,,val,8,1289,43.064114,86.483171,0.894339,0.894339,0.833565
3,exogenous,ridge_pooled_log1p,,val,ALL,4370,17.044272,47.894706,0.932200,0.932200,0.881457


### 6.3 Keep predictions for plots later

In [42]:
ridge_preds = {
    "val":  pd.DataFrame({"basin_id": VAL_DF["basin_id"].values,
                          "date_local": VAL_DF["date_local"].values,
                          "y_true": yva_exog,
                          "yhat_ridge": yhat_val_ridge}),
    "test": pd.DataFrame({"basin_id": TEST_DF["basin_id"].values,
                          "date_local": TEST_DF["date_local"].values,
                          "y_true": yte_exog,
                          "yhat_ridge": yhat_test_ridge}),
}


## Step 7: pooled tree-ensemble model

### 7.1 Pick a backend (LGBM → XGB → RF)

In [43]:
# --- Step 7.1: Choose tree backend (lgbm -> xgb -> rf) ---

TREE_BACKEND = None
try:
    import lightgbm as lgb
    TREE_BACKEND = "lgbm"
    print("Using LightGBM.")
except Exception:
    try:
        import xgboost as xgb
        TREE_BACKEND = "xgb"
        print("LightGBM not available; using XGBoost.")
    except Exception:
        from sklearn.ensemble import RandomForestRegressor
        TREE_BACKEND = "rf"
        print("LightGBM/XGBoost not available; using RandomForest as fallback.")


LightGBM/XGBoost not available; using RandomForest as fallback.


### 7.2 Train pooled tree on log1p(y) and choose best via VAL NSE

In [44]:
# --- Step 7.2: Train pooled tree model on log1p target & select by VAL NSE ---

import numpy as np
import pandas as pd

def train_tree_log1p(
    Xtr, ytr, Xva, yva,
    backend=TREE_BACKEND,
    random_state=42
):
    ytr_log = np.log1p(np.clip(ytr, 0, None))
    yva_true = yva.astype(float)

    candidates = []  # (name, model, val_nse)

    if backend == "lgbm":
        # small grid; early stopping on VAL
        grids = [
            dict(num_leaves=31, learning_rate=0.1, n_estimators=800),
            dict(num_leaves=31, learning_rate=0.05, n_estimators=1200),
            dict(num_leaves=15, learning_rate=0.1, n_estimators=800),
        ]
        for i, p in enumerate(grids):
            mdl = lgb.LGBMRegressor(
                num_leaves=p["num_leaves"],
                learning_rate=p["learning_rate"],
                n_estimators=p["n_estimators"],
                subsample=0.9, colsample_bytree=0.9,
                reg_alpha=0.0, reg_lambda=0.0,
                random_state=random_state, n_jobs=-1
            )
            mdl.fit(
                Xtr, ytr_log,
                eval_set=[(Xva, ytr_log[:len(Xva)])],  # we just need callbacks; metric on log target
                eval_metric="l2",
                callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)],
            )
            yva_hat = np.expm1(mdl.predict(Xva, num_iteration=mdl.best_iteration_))
            yva_hat = np.clip(yva_hat, 0, None)
            candidates.append((f"lgbm_{i}", mdl, nse(yva_true, yva_hat)))

    elif backend == "xgb":
        grids = [
            dict(max_depth=6, learning_rate=0.1, n_estimators=600),
            dict(max_depth=5, learning_rate=0.05, n_estimators=1000),
        ]
        for i, p in enumerate(grids):
            mdl = xgb.XGBRegressor(
                max_depth=p["max_depth"], learning_rate=p["learning_rate"],
                n_estimators=p["n_estimators"], subsample=0.9, colsample_bytree=0.9,
                objective="reg:squarederror", reg_alpha=0.0, reg_lambda=0.0,
                random_state=random_state, n_jobs=-1
            )
            mdl.fit(
                Xtr, ytr_log,
                eval_set=[(Xva, ytr_log[:len(Xva)])],
                verbose=False
            )
            yva_hat = np.expm1(mdl.predict(Xva))
            yva_hat = np.clip(yva_hat, 0, None)
            candidates.append((f"xgb_{i}", mdl, nse(yva_true, yva_hat)))

    else:  # rf fallback
        from sklearn.ensemble import RandomForestRegressor
        grids = [
            dict(n_estimators=600, max_depth=None),
            dict(n_estimators=600, max_depth=14),
        ]
        for i, p in enumerate(grids):
            mdl = RandomForestRegressor(
                n_estimators=p["n_estimators"], max_depth=p["max_depth"],
                random_state=random_state, n_jobs=-1
            )
            mdl.fit(Xtr, ytr_log)
            yva_hat = np.expm1(mdl.predict(Xva))
            yva_hat = np.clip(yva_hat, 0, None)
            candidates.append((f"rf_{i}", mdl, nse(yva_true, yva_hat)))

    # pick best
    best = max(candidates, key=lambda t: (t[2] if not np.isnan(t[2]) else -np.inf))
    return {"name": best[0], "model": best[1], "val_nse": best[2], "backend": backend}

tree_pack = train_tree_log1p(Xtr_exog, ytr_exog, Xva_exog, yva_exog, backend=TREE_BACKEND)
print(f"Tree backend: {tree_pack['backend']}  |  chosen: {tree_pack['name']}  |  VAL NSE: {tree_pack['val_nse']:.3f}")


Tree backend: rf  |  chosen: rf_0  |  VAL NSE: 0.929


### 7.3 Evaluate pooled tree model on VAL & TEST

In [45]:
# --- Step 7.3: Evaluate pooled tree model (VAL & TEST) ---

def predict_with_tree(tree_pack, X):
    yhat = tree_pack["model"].predict(X)
    if yhat.ndim > 1:  # some libs return 2D
        yhat = yhat.ravel()
    yhat = np.expm1(yhat)
    return np.clip(yhat, 0, None)

yhat_val_tree  = predict_with_tree(tree_pack, Xva_exog)
yhat_test_tree = predict_with_tree(tree_pack, Xte_exog)

tree_metrics_val  = metrics_from_arrays(VAL_DF,  yva_exog,  yhat_val_tree,
                                        "exogenous", f"{tree_pack['backend']}_pooled_log1p", "val")
tree_metrics_test = metrics_from_arrays(TEST_DF, yte_exog, yhat_test_tree,
                                        "exogenous", f"{tree_pack['backend']}_pooled_log1p", "test")
tree_metrics = pd.concat([tree_metrics_val, tree_metrics_test], ignore_index=True)

display(tree_metrics.sort_values(["split","basin_id"]))


,track,model,baseline,split,basin_id,n,mae,rmse,r2,nse,kge
4,exogenous,rf_pooled_log1p,,test,3,1289,9.025846,15.174165,0.882067,0.882067,0.882641
5,exogenous,rf_pooled_log1p,,test,6,1796,5.465199,8.510107,0.800770,0.800770,0.761376
6,exogenous,rf_pooled_log1p,,test,8,1290,59.369466,98.710706,0.876898,0.876898,0.916510
7,exogenous,rf_pooled_log1p,,test,ALL,4375,22.408326,54.503212,0.929774,0.929774,0.917808
0,exogenous,rf_pooled_log1p,,val,3,1287,7.672906,15.494944,0.887047,0.887047,0.892494
1,exogenous,rf_pooled_log1p,,val,6,1794,5.259741,8.783014,0.782209,0.782209,0.716939
2,exogenous,rf_pooled_log1p,,val,8,1289,43.268235,88.379874,0.889654,0.889654,0.882037
3,exogenous,rf_pooled_log1p,,val,ALL,4370,17.181638,49.054617,0.928876,0.928876,0.920113


### 7.4 Keep predictions

In [47]:
tree_preds = {
    "val":  pd.DataFrame({"basin_id": VAL_DF["basin_id"].values,
                          "date_local": VAL_DF["date_local"].values,
                          "y_true": yva_exog,
                          "yhat_tree": yhat_val_tree}),
    "test": pd.DataFrame({"basin_id": TEST_DF["basin_id"].values,
                          "date_local": TEST_DF["date_local"].values,
                          "y_true": yte_exog,
                          "yhat_tree": yhat_test_tree}),
}


## A comparison among methods so far

In [49]:
# === Clean comparison: handle 'ALL' separately to avoid str/int sorting ===
import numpy as np
import pandas as pd

def _unify_metrics(baseline_metrics, ridge_metrics, tree_metrics):
    m0 = baseline_metrics.copy()
    m1 = ridge_metrics.copy()
    m2 = tree_metrics.copy()
    for m in (m0, m1, m2):
        if "baseline" not in m.columns:
            m["baseline"] = ""
        if "model" not in m.columns:
            m["model"] = ""
    allm = pd.concat([m0, m1, m2], ignore_index=True)
    allm["algo"] = np.where(allm["model"].eq("baseline"), allm["baseline"], allm["model"])
    keep = ["split","basin_id","algo","n","mae","rmse","r2","nse","kge"]
    return allm[keep].copy()

MET_all = _unify_metrics(baseline_metrics_exog, ridge_metrics, tree_metrics)

# Split per-basin vs overall and coerce per-basin to int
PER = MET_all[MET_all["basin_id"] != "ALL"].copy()
PER["basin_id"] = pd.to_numeric(PER["basin_id"], errors="coerce").astype("Int64")
PER = PER.dropna(subset=["basin_id"]).copy()
PER["basin_id"] = PER["basin_id"].astype(int)

OVR = MET_all[MET_all["basin_id"] == "ALL"].copy()

# --- Pivots: NSE by basin × algo (VAL/TEST) ---
val_nse  = (PER.query("split == 'val'")
              .pivot_table(index="basin_id", columns="algo", values="nse", aggfunc="first")
              .sort_index())
test_nse = (PER.query("split == 'test'")
              .pivot_table(index="basin_id", columns="algo", values="nse", aggfunc="first")
              .sort_index())
print("=== VAL NSE (higher is better) ===");  display(val_nse)
print("=== TEST NSE (higher is better) ==="); display(test_nse)

# --- Skill vs persistence ---
pers = (PER.query("algo == 'persistence'")
          [["split","basin_id","rmse","nse"]]
          .rename(columns={"rmse":"rmse_pers","nse":"nse_pers"}))

CMP = (PER.query("algo != 'persistence'")
         .merge(pers, on=["split","basin_id"], how="left"))
CMP["skill_rmse"] = 1 - (CMP["rmse"] / CMP["rmse_pers"])
CMP["skill_nse"]  = CMP["nse"] - CMP["nse_pers"]

print("=== Skill vs Persistence — VAL ===")
display(CMP.query("split == 'val'")
        [["basin_id","algo","nse","rmse","skill_nse","skill_rmse"]]
        .sort_values(["basin_id","skill_nse"], ascending=[True, False]))

print("=== Skill vs Persistence — TEST ===")
display(CMP.query("split == 'test'")
        [["basin_id","algo","nse","rmse","skill_nse","skill_rmse"]]
        .sort_values(["basin_id","skill_nse"], ascending=[True, False]))

# --- Champion per basin from VAL (max NSE; tie -> lower RMSE) ---
val_ranked = (CMP.query("split == 'val'")
                .sort_values(["basin_id","nse","rmse"], ascending=[True, False, True]))
champ = (val_ranked.groupby("basin_id", as_index=False)
                    .first()[["basin_id","algo","nse","rmse","skill_nse","skill_rmse"]])
print("=== Champions per basin (chosen on VAL NSE) ==="); display(champ)

# --- TEST metrics for champions + skill vs persistence (TEST) ---
test_rows = (PER.query("split == 'test'")
               .merge(champ[["basin_id","algo"]], on=["basin_id","algo"], how="inner"))
test_rows = (test_rows
             .merge(pers.query("split == 'test'")[["basin_id","rmse_pers","nse_pers"]],
                    on="basin_id", how="left"))
test_rows["skill_rmse"] = 1 - (test_rows["rmse"] / test_rows["rmse_pers"])
test_rows["skill_nse"]  = test_rows["nse"] - test_rows["nse_pers"]
print("=== TEST metrics for champions (with skill) ===")
display(test_rows[["basin_id","algo","n","mae","rmse","r2","nse","kge","skill_rmse","skill_nse"]]
        .sort_values("basin_id"))

# --- Overall (ALL) rows, kept separate to avoid str/int mixing ---
print("=== Overall (ALL basins) — VAL ===")
display(OVR.query("split == 'val'")[["algo","nse","rmse"]].sort_values("nse", ascending=False))
print("=== Overall (ALL basins) — TEST ===")
display(OVR.query("split == 'test'")[["algo","nse","rmse"]].sort_values("nse", ascending=False))


=== VAL NSE (higher is better) ===


algo,climatology,persistence,rf_pooled_log1p,ridge_pooled_log1p
basin_id,,,,
3,0.866820,0.951545,0.887047,0.909473
6,0.772489,0.954586,0.782209,0.786226
8,0.838289,0.927352,0.889654,0.894339


=== TEST NSE (higher is better) ===


algo,climatology,persistence,rf_pooled_log1p,ridge_pooled_log1p
basin_id,,,,
3,0.850746,0.939994,0.882067,0.866191
6,0.769334,0.939907,0.800770,0.804489
8,0.805534,0.914919,0.876898,0.894951


=== Skill vs Persistence — VAL ===


...

,basin_id,algo,n,mae,rmse,r2,nse,kge,skill_rmse,skill_nse
0,3,ridge_pooled_log1p,1289,9.765049,16.163297,0.866191,0.866191,0.797522,-0.493294,-0.073803
1,6,ridge_pooled_log1p,1796,4.948941,8.430321,0.804489,0.804489,0.775466,-0.803747,-0.135419
2,8,ridge_pooled_log1p,1290,52.420569,91.185828,0.894951,0.894951,0.935221,-0.111167,-0.019968


=== Overall (ALL basins) — VAL ===


,algo,nse,rmse
11,persistence,0.954075,39.418165
19,ridge_pooled_log1p,0.932200,47.894706
27,rf_pooled_log1p,0.928876,49.054617
3,climatology,0.896762,59.100670


=== Overall (ALL basins) — TEST ===


,algo,nse,rmse
15,persistence,0.952030,45.046164
23,ridge_pooled_log1p,0.939531,50.575115
31,rf_pooled_log1p,0.929774,54.503212
7,climatology,0.889863,68.255688


In [51]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path


FIG_DIR = PROJECT_ROOT / "data/modeling/reports/figs";

FIG_DIR.mkdir(parents=True, exist_ok=True)

# Build VAL/TEST bar charts from PER (per-basin rows only)
def plot_nse_bars(PER, split, outname):
    df = (PER.query("split == @split")
            .pivot_table(index="basin_id", columns="algo", values="nse", aggfunc="first")
            .sort_index())
    # Keep a consistent column order if present
    col_order = [c for c in ["climatology","persistence","ridge_pooled_log1p","rf_pooled_log1p"] if c in df.columns]
    df = df[col_order]

    ax = df.plot(kind="bar", figsize=(9,4), width=0.8)
    ax.set_title(f"NSE by Basin — {split.upper()}")
    ax.set_ylabel("NSE (↑ better)")
    ax.set_xlabel("Basin ID")
    ax.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    outfile = FIG_DIR / outname
    plt.savefig(outfile, dpi=180)
    plt.close()
    print("Wrote", outfile)

plot_nse_bars(PER, "val",  "nse_by_basin_val.png")
plot_nse_bars(PER, "test", "nse_by_basin_test.png")


Wrote /Users/liuq13/bhutan_climate_modeling/data/modeling/reports/figs/nse_by_basin_val.png
Wrote /Users/liuq13/bhutan_climate_modeling/data/modeling/reports/figs/nse_by_basin_test.png


In [52]:
import numpy as np
import matplotlib.pyplot as plt

# Helper: predict with ridge (your champs are ridge pooled)
def _predict_ridge_pack(ridge_pack, X):
    Xs = ridge_pack["scaler"].transform(X)
    yhat = np.expm1(ridge_pack["model"].predict(Xs))
    return np.clip(yhat, 0, None)

def hydrograph_test_for_basin(basin_id, ridge_pack, TEST_DF, Xte, yte, outname, year=None):
    m = (TEST_DF["basin_id"] == basin_id)
    df = TEST_DF.loc[m, ["date_local"]].copy()
    df["y"]  = yte[m].astype(float)
    df["yh"] = _predict_ridge_pack(ridge_pack, Xte[m])

    if year is not None:
        df = df[df["date_local"].dt.year == int(year)]
        suffix = f" (year {year})"
    else:
        suffix = ""

    plt.figure(figsize=(10,3.5))
    plt.plot(df["date_local"], df["y"],  label="Observed", lw=1.4)
    plt.plot(df["date_local"], df["yh"], label="Model",    lw=1.2)
    plt.title(f"Hydrograph — Basin {basin_id}{suffix}")
    plt.ylabel("Discharge (m³/s)")
    plt.xlabel("Date")
    plt.legend(loc="upper right")
    plt.tight_layout()
    outfile = FIG_DIR / outname
    plt.savefig(outfile, dpi=180)
    plt.close()
    print("Wrote", outfile)

# Draw for your three basins; use last test year for a clean story
test_year = TEST_DF["date_local"].dt.year.max()
for b in sorted(champ["basin_id"].unique()):
    hydrograph_test_for_basin(
        basin_id=b, ridge_pack=ridge_search,
        TEST_DF=TEST_DF, Xte=Xte_exog, yte=yte_exog,
        outname=f"hydrograph_basin{b}_{test_year}.png",
        year=test_year
    )


Wrote /Users/liuq13/bhutan_climate_modeling/data/modeling/reports/figs/hydrograph_basin3_2024.png
Wrote /Users/liuq13/bhutan_climate_modeling/data/modeling/reports/figs/hydrograph_basin6_2024.png
Wrote /Users/liuq13/bhutan_climate_modeling/data/modeling/reports/figs/hydrograph_basin8_2024.png


In [53]:
import matplotlib.pyplot as plt

def residual_box_by_month(basin_id, ridge_pack, TEST_DF, Xte, yte, outname):
    m = (TEST_DF["basin_id"] == basin_id)
    df = TEST_DF.loc[m, ["date_local"]].copy()
    yh = _predict_ridge_pack(ridge_pack, Xte[m])
    df["resid"] = yte[m] - yh
    df["month"] = df["date_local"].dt.month

    plt.figure(figsize=(8,3.5))
    df.boxplot(column="resid", by="month", grid=False)
    plt.suptitle("")
    plt.title(f"Residuals by Month — Basin {basin_id}")
    plt.xlabel("Month"); plt.ylabel("Observed − Predicted (m³/s)")
    plt.tight_layout()
    outfile = FIG_DIR / outname
    plt.savefig(outfile, dpi=180)
    plt.close()
    print("Wrote", outfile)

for b in sorted(champ["basin_id"].unique()):
    residual_box_by_month(
        basin_id=b, ridge_pack=ridge_search,
        TEST_DF=TEST_DF, Xte=Xte_exog, yte=yte_exog,
        outname=f"residuals_by_month_basin{b}.png"
    )


Wrote /Users/liuq13/bhutan_climate_modeling/data/modeling/reports/figs/residuals_by_month_basin3.png
Wrote /Users/liuq13/bhutan_climate_modeling/data/modeling/reports/figs/residuals_by_month_basin6.png
Wrote /Users/liuq13/bhutan_climate_modeling/data/modeling/reports/figs/residuals_by_month_basin8.png


<Figure size 800x350 with 0 Axes>

<Figure size 800x350 with 0 Axes>

<Figure size 800x350 with 0 Axes>